## Paso 0 — Cargamos las 18 tablas ya tipadas

Mismo patrón que los notebooks anteriores: leemos directo los Parquet
del notebook 00.


In [1]:
import pandas as pd
from pathlib import Path

RUTA_INTERIM = Path("../datos/interim")
ARCHIVOS_PARQUET = sorted(RUTA_INTERIM.glob("*.parquet"))
tablas = {archivo.stem: pd.read_parquet(archivo) for archivo in ARCHIVOS_PARQUET}
assert len(tablas) == 18, f"Se esperaban 18 tablas, se cargaron {len(tablas)}"
print(f"Tablas cargadas: {len(tablas)}")


Tablas cargadas: 18


## Paso 1 — ¿Cuántas líneas Tarifario tienen suficiente info para comparar contra SAP? (§7.2 del plan)

Para comparar el precio cotizado contra el precio de mercado SAP,
necesitamos saber qué LineaNegocio/ClaseViaje usó esa línea — dato que
viene de lineas_solicitud, no de cotizacion_lineas directamente. Ya
sabíamos que esto casi nunca se llena (notebook 01) — aquí confirmamos
cuántas de las 32 líneas Tarifario específicamente tienen ese dato.


In [2]:
cotizacion_lineas = tablas["cotizacion_lineas"]
lineas_solicitud = tablas["lineas_solicitud"]

tarifario = cotizacion_lineas[cotizacion_lineas["fuente"] == "Tarifario"].copy()
tarifario = tarifario.merge(
    lineas_solicitud[["id", "lineaNegocio", "claseViaje", "rutaId"]],
    left_on="lineaSolicitudId", right_on="id", suffixes=("", "_linea")
)

print(f"Líneas con Fuente=Tarifario: {len(cotizacion_lineas[cotizacion_lineas['fuente']=='Tarifario'])}")
con_ambos = tarifario.dropna(subset=["lineaNegocio", "claseViaje"])
print(f"De esas, con LineaNegocio Y ClaseViaje pobladas: {len(con_ambos)}")
print()
print(con_ambos[["id", "lineaNegocio", "claseViaje", "rutaId", "valorFlete"]].to_string())


Líneas con Fuente=Tarifario: 32
De esas, con LineaNegocio Y ClaseViaje pobladas: 0

Empty DataFrame
Columns: [id, lineaNegocio, claseViaje, rutaId, valorFlete]
Index: []


## Paso 1b — Cruce real por ruta normalizada contra SAP (§7.1/§7.2 del plan, resuelto)

En el Paso 1 concluimos que cruzar por `LineaNegocio`/`ClaseViaje` da 0
casos usables. Pero en el notebook 06 (Fase 8) sí resolvimos la ruta con
`rapidfuzz` — volvemos aquí a cerrar el ciclo: para las 32 líneas
`Fuente=Tarifario`, resolvemos su ruta (vía `lineaSolicitudId` →
`lineas_solicitud` → `rutaId` → `catalogo_rutas`) a la zona SAP
correspondiente, y comparamos el precio cotizado contra el rango real
de `tarifa-plaza` para esa zona — **excluyendo primero las tarifas de
"Devolución Contenedor"** (`lineanegocio="6"`): es la tarifa de
reposicionar un contenedor vacío, no de transportar carga, así que
mezclarla en el mismo mínimo/mediana/máximo compara servicios distintos.
Sigue sin filtrar por tipo de vehículo (no tenemos esa traducción), así
que el rango de comparación es más amplio de lo ideal, no una
comparación estrictamente pareja.

In [3]:
import re
import unicodedata
from rapidfuzz import process, fuzz

def limpiar_nombre_ciudad(texto):
    if pd.isna(texto):
        return None
    sin_departamento = re.sub(r"\s*\([^)]*\)\s*$", "", texto).strip()
    sin_tildes = unicodedata.normalize("NFKD", sin_departamento).encode("ascii", "ignore").decode("ascii")
    return sin_tildes.upper()

lineas_solicitud = tablas["lineas_solicitud"]
catalogo_rutas = tablas["catalogo_rutas"]
tarifa_plaza = tablas["tarifa-plaza-tblSAPTarifas-2022-09-01_a_hoy"]
historico_sap = tablas["historico-sap-tblSAPTarifasLog-2022-09-01_a_hoy"]

base = cotizacion_lineas.merge(
    lineas_solicitud[["id", "rutaId"]], left_on="lineaSolicitudId", right_on="id", suffixes=("", "_linea")
).merge(
    catalogo_rutas[["id", "origen", "destino"]], left_on="rutaId", right_on="id", suffixes=("", "_ruta")
)
base["origen_limpio"] = base["origen"].apply(limpiar_nombre_ciudad)
base["destino_limpio"] = base["destino"].apply(limpiar_nombre_ciudad)

# mismo vocabulario y umbral que en el notebook 06 (§9) — sorted() para orden determinista
valores_sap_codigos = (
    set(tarifa_plaza["origen"].dropna().unique()) | set(tarifa_plaza["destino"].dropna().unique())
    | set(historico_sap["origen"].dropna().unique()) | set(historico_sap["destino"].dropna().unique())
)
vocabulario_sap = {re.sub(r"^ZT?C_", "", c).replace("_", " "): c for c in sorted(valores_sap_codigos)}
nombres_sap = list(vocabulario_sap.keys())

def resolver_zona_sap(ciudad, umbral=90):
    if ciudad is None:
        return None
    mejor = process.extractOne(ciudad, nombres_sap, scorer=fuzz.WRatio)
    return vocabulario_sap[mejor[0]] if mejor[1] >= umbral else None

base["zona_origen"] = base["origen_limpio"].apply(resolver_zona_sap)
base["zona_destino"] = base["destino_limpio"].apply(resolver_zona_sap)

tarifario_ruta = base[base["fuente"] == "Tarifario"].copy()

# Corrección post-cierre (revisión de calidad): "Oferta de Expedición Devolución
# Contenedor" (lineanegocio="6") es la tarifa de reposicionar un contenedor vacío, no
# de transportar carga — se excluye de los candidatos antes de comparar, para no diluir
# el mínimo/mediana/máximo con un servicio distinto al que de verdad cotizó CabinaFletes.
candidatos_carga_real = tarifa_plaza[tarifa_plaza["lineanegocio"] != "6"]

filas = []
for _, fila in tarifario_ruta.iterrows():
    candidatos = candidatos_carga_real[
        (candidatos_carga_real["origen"] == fila["zona_origen"]) & (candidatos_carga_real["destino"] == fila["zona_destino"])
    ]
    filas.append({
        "origen": fila["origen"], "destino": fila["destino"], "valorFlete": fila["valorFlete"],
        "n_candidatos_sap": len(candidatos),
        "sap_min": candidatos["valor"].min() if len(candidatos) else None,
        "sap_mediana": candidatos["valor"].median() if len(candidatos) else None,
        "sap_max": candidatos["valor"].max() if len(candidatos) else None,
    })

df_tarifario_ruta = pd.DataFrame(filas)
df_tarifario_ruta["diferencia_pct_vs_mediana"] = (
    (df_tarifario_ruta["valorFlete"] - df_tarifario_ruta["sap_mediana"]) / df_tarifario_ruta["sap_mediana"] * 100
)
df_tarifario_ruta["dentro_de_rango_sap"] = (
    (df_tarifario_ruta["valorFlete"] >= df_tarifario_ruta["sap_min"])
    & (df_tarifario_ruta["valorFlete"] <= df_tarifario_ruta["sap_max"])
)

print(f"Líneas Tarifario con match de ruta: {(df_tarifario_ruta['n_candidatos_sap'] > 0).sum()} de {len(df_tarifario_ruta)}")
print(f"Dentro del rango [min,max] de SAP para esa ruta: {df_tarifario_ruta['dentro_de_rango_sap'].sum()} de {len(df_tarifario_ruta)}")
print()
print(df_tarifario_ruta.to_string())
print()
print("Diferencia % vs. mediana SAP:")
print(df_tarifario_ruta["diferencia_pct_vs_mediana"].describe())

Líneas Tarifario con match de ruta: 32 de 32
Dentro del rango [min,max] de SAP para esa ruta: 32 de 32

          origen       destino  valorFlete  n_candidatos_sap    sap_min  sap_mediana    sap_max  diferencia_pct_vs_mediana  dentro_de_rango_sap
0   Buenaventura        Bogotá   6400000.0                58  1544000.0    4700000.0  7000000.0                  36.170213                 True
1           Cali        Bogotá   2300000.0                17  1650000.0    3361178.0  4915466.0                 -31.571610                 True
2         Bogotá     Cartagena   8650000.0                48  3000000.0    5600000.0  8840928.0                  54.464286                 True
3         Bogotá  Barranquilla   7667624.0                45  2600000.0    4880000.0  7900000.0                  57.123443                 True
4    Bucaramanga        Bogotá   2790000.0                13   900000.0    2500000.0  4064499.0                  11.600000                 True
5           Cali  Buenaventura  

## Paso 1c — Cruce real por ruta normalizada contra SICE-TAC

Mismo ejercicio para las 13 líneas `Fuente=Sicetac`, contra `piso-sicetac`
(núcleo + raras combinadas). Se excluye la línea con la ruta malformada de
prueba ("b"/"j") — buscar una sola letra como substring encuentra falsos
positivos en cualquier ciudad que la contenga.

Dos correcciones de método (revisión post-cierre), igual de importantes
que el cruce en sí: **(1)** `nombreruta` viene siempre como "ORIGEN -
DESTINO" — buscar cada ciudad como substring en cualquier parte del texto
(como se hacía antes) no distingue sentido, y una cotización Medellín→
Bogotá terminaba comparada contra la misma fila que Bogotá→Medellín, sin
verificar que la ruta real fuera en esa dirección. **(2)** `piso-sicetac`
incluye filas `condicioncarga="VACIO"` (tarifa de reposicionar un
vehículo vacío, no de transportar carga real) — no son comparables contra
un `ValorFlete` de una cotización con carga, y había una en el único
candidato encontrado antes. Se corrigen ambas cosas antes de comparar.

In [4]:
sicetac_todo = pd.concat([tablas["piso-sicetac_nucleo"], tablas["piso-sicetac_raras"]], ignore_index=True)

# (1) nombreruta = "ORIGEN - DESTINO" en el 100% de las filas (verificado) — partimos el
# texto para exigir que cada ciudad esté en su lado correcto, no en cualquier parte.
sicetac_todo[["origen_ruta_txt", "destino_ruta_txt"]] = sicetac_todo["nombreruta"].str.split(" - ", n=1, expand=True)

# (2) condicioncarga="VACIO" (nombretipocarga="Sin carga") es tarifa de vehículo vacío,
# no de carga real — se excluye antes de buscar candidatos comparables.
sicetac_carga_real = sicetac_todo[sicetac_todo["condicioncarga"] == "CARGADO"]

sicetac_ruta = base[(base["fuente"] == "Sicetac") & (base["origen_limpio"].str.len() > 2)].copy()

filas = []
for _, fila in sicetac_ruta.iterrows():
    candidatos = sicetac_carga_real[
        sicetac_carga_real["origen_ruta_txt"].str.contains(fila["origen_limpio"], na=False)
        & sicetac_carga_real["destino_ruta_txt"].str.contains(fila["destino_limpio"], na=False)
    ]
    filas.append({
        "origen": fila["origen"], "destino": fila["destino"], "valorFlete": fila["valorFlete"],
        "n_candidatos_sicetac": len(candidatos),
        "sicetac_valor": candidatos["valorsiceconhora"].median() if len(candidatos) else None,
    })

df_sicetac_ruta = pd.DataFrame(filas)
df_sicetac_ruta["diferencia_pct"] = (
    (df_sicetac_ruta["valorFlete"] - df_sicetac_ruta["sicetac_valor"]) / df_sicetac_ruta["sicetac_valor"] * 100
)

print(f"Líneas Sicetac válidas (excluida la malformada): {len(sicetac_ruta)}")
print(f"Con al menos 1 candidato SICE-TAC de carga real, misma dirección: {(df_sicetac_ruta['n_candidatos_sicetac'] > 0).sum()}")
print()
print(df_sicetac_ruta.to_string())

Líneas Sicetac válidas (excluida la malformada): 12
Con al menos 1 candidato SICE-TAC de carga real, misma dirección: 0

          origen       destino  valorFlete  n_candidatos_sicetac sicetac_valor diferencia_pct
0       Medellin        Bogota   4832788.0                     0          None            NaN
1       Medellin          Cali   1802013.0                     0          None            NaN
2         Bogota  Barranquilla   8175231.0                     0          None            NaN
3         Bogotá      Medellín   1564425.0                     0          None            NaN
4   Barranquilla        Bogotá   2818633.0                     0          None            NaN
5   Barranquilla        Bogotá   2818633.0                     0          None            NaN
6         Bogotá      Medellín   1564425.0                     0          None            NaN
7   Barranquilla        Bogotá   2818633.0                     0          None            NaN
8         Bogotá      Medellín   

## Paso 2 — Dispersión de VALOR en SAP, por LineaNegocio (§7.3 del plan)

Sin depender del cruce con lo propio: ¿qué tan distinto es el precio de
una tarifa SAP según la categoría de negocio? Esto caracteriza el
mercado por sí mismo, información con valor propio para el futuro
modelo, independiente de si se puede cruzar contra CabinaFletes o no.


In [5]:
tarifa_plaza = tablas["tarifa-plaza-tblSAPTarifas-2022-09-01_a_hoy"]

por_linea_negocio = tarifa_plaza.groupby("lineaNegocioDescripcion")["valor"].describe()
print(por_linea_negocio.to_string())



                                              count          mean           std        min        25%        50%        75%         max
lineaNegocioDescripcion                                                                                                                
Oferta de Expedición Devolución Contenedor   1028.0  1.863085e+06  1.439189e+06        0.0   900000.0  1300001.0  2500000.0   8694464.0
Oferta de Expedición Interior a Puerto       2340.0  4.041846e+06  2.270570e+06   220000.0  2232500.0  3600000.0  5500000.0  13272000.0
Oferta de Expedición Licitaciones               1.0  3.556418e+06           NaN  3556418.0  3556418.0  3556418.0  3556418.0   3556418.0
Oferta de Expedición Nacional               13625.0  3.163931e+06  2.038501e+06   100000.0  1600000.0  2700000.0  4200000.0  18000000.0
Oferta de Expedición Puerto a interior       6023.0  4.030362e+06  2.411012e+06   100000.0  2000000.0  3600000.0  5600000.0  15700000.0


## Paso 3 — Dispersión de VALOR en SAP, por tipo de vehículo (§7.3 del plan)

Mismo ejercicio que LineaNegocio, ahora por TIPOVEHICULO (7 categorías,
ya confirmadas 1 a 1 con su descripción en el notebook 04). Misma
salvedad: mezcla variación real de vehículo con variación de ruta.


In [6]:
por_tipo_vehiculo = tarifa_plaza.groupby("vehiculoDescripcion")["valor"].describe()
print(por_tipo_vehiculo.to_string())


                                           count          mean           std       min        25%        50%        75%         max
vehiculoDescripcion                                                                                                                
Dobletroque                                438.0  3.641868e+06  1.390625e+06  700000.0  2800000.0  3500000.0  4600000.0   8000000.0
Liviano (capacidad 4 ton)                 5221.0  1.826780e+06  9.436562e+05  220000.0  1100000.0  1600000.0  2500000.0   7500000.0
Liviano mas 10.5 ton (capacidad 8.5 ton)  5147.0  2.377126e+06  1.191969e+06  100000.0  1400000.0  2200000.0  3224000.0   8500000.0
Mini Mula Poderosa                         421.0  4.686166e+06  1.780740e+06  700000.0  3500000.0  4760000.0  5800000.0   8800000.0
Minimula                                  3488.0  4.087465e+06  1.886331e+06  570000.0  2700000.0  4000000.0  5200000.0  12500000.0
Rígidos, Turbo o Sencillo                   14.0  2.315387e+06  1.371188e+06

## Paso 4 — Dispersión de VALORSICECONHORA en SICE-TAC, por tipo de carga (§7.3 del plan)

Solo sobre `piso-sicetac_nucleo` (126 filas) — las 25 `raras` se dejan
fuera a propósito, son casos extremos elegidos adrede, mezclarlas
sesgaría el resultado hacia arriba/abajo artificialmente.


In [7]:
piso_sicetac_nucleo = tablas["piso-sicetac_nucleo"]
por_tipo_carga = piso_sicetac_nucleo.groupby("nombretipocarga")["valorsiceconhora"].describe()
print(por_tipo_carga.to_string())


                    count          mean           std        min         25%        50%         75%        max
nombretipocarga                                                                                               
Carga Refrigerada     6.0  2.648267e+06  1.502266e+06   913366.0  1434634.25  2619199.5  3617280.25  4748086.0
Contenedor cargado    6.0  4.113858e+06  2.719678e+06  1474516.0  2141309.75  3294614.5  5678825.25  8388967.0
Contenedor vacío      6.0  2.042604e+06  1.682862e+06   330531.0   930772.50  1921094.0  2300937.25  5054689.0
General              24.0  2.568956e+06  1.383970e+06   705009.0  1484076.75  2224079.5  3426629.00  5406168.0
Granel Sólido        30.0  3.050617e+06  1.646392e+06   367451.0  1743835.00  3197555.0  4459708.25  6251385.0
Granel líquido        6.0  4.115099e+06  3.157665e+06  1017941.0  1952659.25  3267965.0  5408030.75  9501111.0
Sin carga            48.0  1.668203e+06  1.366734e+06    39366.0   549494.25  1258556.5  2307198.00  6060042.0


## Paso 5 — Fuente × Resultado de la cotización (§7.4 del plan)

Modalidad × líneas de negocio SAP no se puede — Modalidad tiene solo 1
valor real en las 13 solicitudes. Este otro cruce sí es posible:
¿la fuente que usó CabinaFletes para cotizar tiene alguna relación con
si la cotización se ganó o no? Con 18 cotizaciones, el plan mismo avisa
que cualquier patrón aquí es anecdótico, no estadístico — se reporta
igual, con esa salvedad explícita.


In [8]:
cotizaciones = tablas["cotizaciones"]
cl_con_resultado = cotizacion_lineas.merge(
    cotizaciones[["id", "resultado"]], left_on="cotizacionId", right_on="id", suffixes=("", "_cot")
)
tabla_cruzada = pd.crosstab(cl_con_resultado["fuente"], cl_con_resultado["resultado"])
print(tabla_cruzada)


resultado  Ganada  Perdida  SinDefinir
fuente                                
Manual          2        0           1
Sicetac         2        2           9
Tarifario       8        8          16


## Síntesis del notebook 05 — Bivariado y multivariado (Fase 6, §7 del plan)

**Objetivo del plan:** responder qué tan cerca cotiza CabinaFletes del
piso/plaza de mercado — el corazón de esta HU. Resultado: **se pudo
responder para SAP; para SICE-TAC, la respuesta honesta es que esta
muestra no lo permite** (ver corrección de método más abajo).

**Primer intento, por código de negocio (§7.1/§7.2, Paso 1): 0 casos
utilizables.** Solo 5 líneas de solicitud tienen `LineaNegocio`/
`ClaseViaje` poblados; de esas, solo 3 llegaron a cotizarse — y las 3
resultan ser casos ya conocidos como ruido (la ruta malformada
`"b"/"j"`, y los 2 valores `Fuente=Manual` ya descartados por absurdos).
Comparar solo por código de negocio (sin ruta) tampoco habría sido
válido de todas formas — la misma combinación de `LineaNegocio`/
`ClaseViaje` cubre miles de rutas SAP con precios muy distintos entre sí.

**Segundo intento, por ruta normalizada (Pasos 1b-1c).** Una vez resuelta
la ruta con `rapidfuzz` (notebook 06, Fase 8), se regresó aquí a cerrar
el ciclo:
- **Contra SAP (Tarifario, 32 de 32 líneas):** las 32 líneas encuentran
  candidatos de precio para su ruta (13-58 tarifas candidatas según
  cuántos vehículos/pesos aplican, **ya excluidas las tarifas de
  "Devolución Contenedor"** — reposicionar un contenedor vacío no es
  comparable con transportar carga) y las 32 caen dentro del rango
  observado [mínimo, máximo] de SAP para esa ruta. Patrón: CabinaFletes
  cotiza consistentemente **por encima de la mediana** de SAP para la
  misma ruta (mediana de diferencia +30,6%, hasta +57% en
  Bogotá-Barranquilla), con la excepción de Cali-Bogotá que cotiza 31,6%
  por debajo. **Salvedad:** la comparación es contra el rango completo de
  tipos de vehículo de esa ruta (no tenemos traducción de tipo de
  vehículo propio→SAP), así que no es una comparación estrictamente
  pareja vehículo-por-vehículo.
- **Contra SICE-TAC (Sicetac): 0 de 12 líneas válidas encuentran
  candidato comparable — corregido tras revisión de calidad.** La
  primera versión de esta celda reportaba 5/12 con un hallazgo llamativo
  (+43% en una dirección, -54% "por debajo del piso legal" en la
  contraria). Al revisar la evidencia, el único candidato usado para
  ambas direcciones resultó ser una fila `condicioncarga="VACIO"`
  ("Sin carga" — tarifa de reposicionar un vehículo vacío, no de
  transportar carga real) — no es comparable contra un `ValorFlete` de
  una cotización con carga. Además, la búsqueda no distinguía sentido de
  ruta, así que Medellín→Bogotá y Bogotá→Medellín se comparaban contra
  la misma fila sin verificar la dirección. Corregidas ambas cosas
  (exigir `condicioncarga="CARGADO"` y respetar "ORIGEN - DESTINO"), el
  resultado real es que **ninguna de las 12 líneas encuentra un
  candidato de carga real en la misma dirección** dentro de la muestra
  de 151 filas — consistente con el hallazgo ya documentado en el
  notebook 03 (H-007: `Fuente=Sicetac` no verificable con este extracto,
  la muestra no fue diseñada para incluir estas rutas). No hay evidencia,
  ni a favor ni en contra, de que CabinaFletes esté cotizando por debajo
  del piso SICE-TAC — simplemente no se puede comparar con este extracto.

**Mercado por su cuenta, sin depender de lo propio (§7.3, Pasos 2-4):**
- SAP por `LineaNegocio`: `Nacional` domina (59% del catálogo) y es la
  más barata en promedio; `Puerto a Interior`/`Interior a Puerto` casi
  idénticas entre sí (razonable, son la misma ruta en direcciones
  opuestas); `Licitaciones` tiene solo 1 fila en todo el catálogo.
- SAP por tipo de vehículo: patrón sensato — el precio sube con la
  capacidad (`Liviano 4 ton` la más barata, `Tractomula` la más cara).
  `Rígidos, Turbo o Sencillo` con solo 14 filas, muestra chica.
- SICE-TAC núcleo por tipo de carga: `Sin carga` la más barata,
  `Contenedor cargado`/`Granel líquido` las más caras — también sensato.
  Varias categorías con solo 6 filas cada una (recordar el diseño del
  núcleo: 6 filas por combinación).
- Salvedad transversal: estas dispersiones mezclan variación de negocio
  con variación de ruta — no se pueden separar sin el trabajo de la
  Fase 8.

**Categórica × categórica (§7.4, Paso 5):** `Modalidad` × líneas de
negocio SAP no aplica (Modalidad tiene solo 1 valor real). `Fuente` ×
`Resultado`: calculado (Tarifario y Sicetac ambos 50/50 Ganada/Perdida),
pero **declarado sin señal de negocio** — el usuario confirmó que
`Resultado` también lo marcó él mismo durante pruebas, no refleja una
decisión real de cliente. Se documenta como "no aplicable para inferir
negocio", no como patrón débil.

**Multivariado (§7.5):** no aplicable, tal como anticipaba el plan —
forzar clustering/PCA sobre unas pocas decenas de casos sería
sobreajustar a ruido.

**Qué le queda listo al siguiente notebook:** un hallazgo real y
reproducible para SAP/Tarifario (cotiza ~30% sobre la mediana de
mercado), y una reconfirmación — con método más riguroso — de que el
cruce fino contra SICE-TAC no es posible con esta muestra de 151 filas.
Ambos son insumo directo para `hallazgos.md`/`limitaciones.md`. Siguen
las Fases 7-11 del plan (temporal, geográfico, outliers, sesgos, vacíos)
antes de la síntesis final.
